# NUMBER SUMS

## IMPORTING LIBRARIES

In this game, you remove cells until the remaining numbers match the targets shown to the right of each row and below each column. In the interactive game, the coordinates range from **1 to 8**. Before running the game, install the **Number sums** library located in `src/` from the project root:

```bash
pip install .
```

In [1]:
from number_sums import NumberSumsGame
from number_sums import NumberSumsController

from IPython.display import clear_output

## CREATE AND PRINT AN 8×8 BOARD

The fixed seed makes this demonstration reproducible.

In [2]:
game = NumberSumsGame.random(seed=42)

assert game.size == 8
assert len(game.board) == 8
assert all(len(line  ) == 8 for line in game.board)

print(game)

      1  2  3  4  5  6  7  8   meta (atual)
   +-------------------------+
 1 |  2  1  5  4  4  3  2  9 |   15 (  30)
 2 |  2  7  1  1  2  4  4  9 |    9 (  30)
 3 |  1  9  4  9  7  4  8  5 |   26 (  47)
 4 |  1  3  7  6  5  3  4  6 |   19 (  35)
 5 |  2  2  7  2  6  6  5  1 |   11 (  31)
 6 |  8  9  2  7  2  9  5  6 |   29 (  48)
 7 |  4  2  1  4  5  2  4  2 |    7 (  24)
 8 |  7  5  8  6  3  6  6  4 |   16 (  45)
   +-------------------------+
     17 16 12 20  8  9 23 27   metas
     27 38 35 39 34 37 38 42   atuais


## TEST THE SOLVER

The solution is computed from the board and its targets. The test is performed on a second instance to keep the board displayed above unchanged.

In [3]:
solution = game.solve()

verification = NumberSumsGame(
    game.board         ,
    game.row_targets   ,
    game.column_targets,
)
verification.apply_solution(solution)


assert verification.is_won()
assert verification.current_row_sums   () == verification.row_targets
assert verification.current_column_sums() == verification.column_targets


print(f"Test completed: valid solution with {len(solution)} cells removed.")

Test completed: valid solution with 36 cells removed.


## INTERACTIVE GAME

Enter `row column` to remove/restore a cell or `keep row column` to mark/unmark a cell as KEEP. Incompatible moves are blocked. You can also use `hint`, `step`, `undo`, and `quit`.

In [4]:
def clear_screen() -> None:
    """Clears the current Jupyter cell output."""

    clear_output(wait=True)


def play(game: NumberSumsGame, input_fn=input) -> bool:
    """Runs a text-based game validated by the CSP"""

    controller     = NumberSumsController(game)
    status_message = "Remove cells or mark them as KEEP until all targets are reached."

    while not controller.won:
        clear_screen()

        print(game.render())

        removed = [
            f"({cell.row + 1}, {cell.column + 1})={cell.value}"
            for cell in game.removed_cells
        ]
        marked = [
            f"({cell.row + 1}, {cell.column + 1})={cell.value}"
            for cell in game.marked_cells
        ]

        print("Removed:", ", ".join(removed) if removed else "none")
        print("KEEP:   ", ", ".join(marked ) if marked  else "none")
        print(f"Forbidden moves: {controller.forbidden_move_count}")

        if status_message:
            print()
            print(f"{status_message}")

        try:
            entry = input_fn(
                f"\n[1-{game.size}] row column | keep | hint | step | undo | quit: "
            ).strip().lower()
        except (EOFError, KeyboardInterrupt):
            print()
            print("Game ended.")

            return False

        if entry in {"quit", "q"}:
            print("Game ended.")
            return False

        if entry in {"undo", "u"}:
            changed = controller.undo()
            status_message = (
                "Last move undone."
                if changed
                else "There are no moves to undo."
            )

            continue

        if entry in {"hint", "h"}:
            hint = controller.request_hint()
            if hint is None:
                status_message = "No pending move."
            else:
                action = "Remove" if hint.action == "remove" else "Mark as KEEP"
                status_message = (
                    f"Hint: {action} row {hint.row + 1}, column {hint.column + 1} "
                    f"(value {hint.value}). {hint.reason}"
                )

            continue

        if entry in {"step", "s"}:
            changed = controller.apply_hint()
            status_message = "CSP step applied." if changed else "No pending move."

            continue

        parts = entry.replace(",", " ").split()
        mark  = bool(parts and parts[0] in {"keep", "k"})

        if mark:
            parts = parts[1:]

        if len(parts) != 2:
            status_message = "Invalid input. Examples: 3 5 or keep 3 5."
            continue

        try:
            row, column = (int(part) - 1 for part in parts)
        except ValueError:
            status_message = "Row and column must be integers."
            continue

        if not (0 <= row < game.size and 0 <= column < game.size):
            status_message = f"Use values between 1 and {game.size}."
            continue

        forbidden_before = controller.forbidden_move_count
        if mark:
            changed = controller.toggle_mark(row, column)
        else:
            changed = controller.toggle_cell(row, column)

        if changed:
            status_message = "Move applied."
        elif controller.forbidden_move_count > forbidden_before:
            status_message = (
                f"Forbidden move. Total: {controller.forbidden_move_count}."
            )
        else:
            status_message = "Game state unchanged."

    clear_screen()

    print(game.render())

    print()
    print("Congratulations! All targets have been reached."    )
    print(f"Forbidden moves: {controller.forbidden_move_count}")

    return True

Run the cell below **manually** to start a new random game. It will wait for your keyboard input; when using **Run All**, enter the `quit` command if you do not want to play at that moment.

In [5]:
new_game = NumberSumsGame.random()

play(new_game)

      1  2  3  4  5  6  7  8   meta (atual)
   +-------------------------+
 1 |  1  7  4  ·  4  5  ·  · |    7 (  21)
 2 |  2  9  6  3  5  8  5  5 |   22 (  43)
 3 |  3  9  4  1  6  1  1  8 |    6 (  33)
 4 |  3  3  8  8  9  8  8  5 |   33 (  52)
 5 |  9  4  8  6  4  9  3  3 |   16 (  46)
 6 |  1  9  8  6  9  5  2  7 |   11 (  47)
 7 |  8  1  7  2  1  4  5  8 |   17 (  36)
 8 |  8  5  8  3  1  1  2  6 |   13 (  34)
   +-------------------------+
      4 20 31 12 11 31 11  5   metas
     35 47 53 29 39 41 26 42   atuais
Removed: (1, 8)=9, (1, 7)=4, (1, 4)=7
KEEP:    none
Forbidden moves: 0

Move applied.
Game ended.


False